In [ ]:
!nvidia-smi

In [ ]:
import os
import numpy as np
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
from heapq import heappop, heappush
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
import shutil
import os


file_path1 = 'A/Padilla - 1 Nobleza virtuosa_Extract.pdf'
doc_path1 = 'A/Padilla - 1 Nobleza virtuosa_Transcription (1).docx'
file_path2 = 'A/Padilla - 2 Noble perfecto_Extract.pdf'
doc_path2 = 'A/Padilla - 2 Noble perfecto_Transcription.docx'

# extended dataset A

doc_path1 = 'A/Dataset A/PORCONES.228.35 – 1636 transcription.docx'
doc_path2 = 'A/Dataset A/PORCONES.82.4 – 1644 transcription.docx'
doc_path3 = 'A/Dataset A/PORCONES.954.2.45 transcription.docx'

# extended dataset B
dp1 = 'A/Dataset B - different typeface/Buendia transcription.docx'
dp2 = 'A/Dataset B - different typeface/Constituciones sinodales transcription.docx'
dp3 = 'A/Dataset B - different typeface/Covarrubias transcription.docx'
dp4 = 'A/Dataset B - different typeface/Ezcaray transcription.docx'
dp5 = 'A/Dataset B - different typeface/Guardiola transcription.docx'
dp6 = 'A/Dataset B - different typeface/Mendo transcription.docx'
dp7 = 'A/Dataset B - different typeface/Milán transcription.docx'
dp8 = 'A/Dataset B - different typeface/Paredes transcription.docx'
dp9 = 'A/Dataset B - different typeface/R.12175.3 transcription.docx'
dp10 = 'A/Dataset B - different typeface/Recopilacion transcription.docx'

In [ ]:
import fitz
from PIL import Image
import os

def convert_pdf_to_images(pdf_path, output_folder,book):
    doc = fitz.open(pdf_path)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    image_paths = []
    for page_num in range(16):
        page = doc.load_page(page_num)
        pix = page.get_pixmap()
        image_path = os.path.join(output_folder, f"{book}_page_{page_num + 1}.png")
        # image_path = image_path.replace('\\', '/')
        pix.save(image_path)
        image_paths.append(image_path)

    doc.close()
    return image_paths


In [ ]:
pdf_path1 = "A/Padilla - 1 Nobleza virtuosa_Extract.pdf"
image_folder1 = "./content/image_folder1"
book1 = 'Padilla - 1'
pdf_path2 = "A/Padilla - 2 Noble perfecto_Extract.pdf"
image_folder2 = "./content/image_folder2"
book2 = 'Padilla - 2'
image_path1 = convert_pdf_to_images(pdf_path1, image_folder1,book1)
image_path2 = convert_pdf_to_images(pdf_path2, image_folder2,book2)

image_path = []
image_path.extend(image_path1)
image_path.extend(image_path2)
print(image_path)

In [ ]:
# Dataset A
image_path3 = ['/content/image_folder3/PORCONES.82.4 – 1644_page_1.png','/content/image_folder3/PORCONES.82.4 – 1644_page_2.png','/content/image_folder3/PORCONES.82.4 – 1644_page_3.png']
image_path4 = ['/content/image_folder4/PORCONES.954.2.45_page_1.png','/content/image_folder3/PORCONES.954.2.45_page_2.png']
image_path5 = ['/content/image_folder5/PORCONES.228.35 – 1636_page_1.png','/content/image_folder5/PORCONES.228.35 – 1636_page_2.png','/content/image_folder5/PORCONES.228.35 – 1636_page_3.png']

# Dataset B
image_path6 = ['/content/image_folder6/Buendia - Instruccion_page_2.png','/content/image_folder6/Buendia - Instruccion_page_3.png','/content/image_folder6/Buendia - Instruccion_page_4.png']
image_path7 = ['/content/image_folder7/Constituciones sinodales Calahorra 1602_page_2.png','/content/image_folder7/Constituciones sinodales Calahorra 1602_page_3.png','/content/image_folder7/Constituciones sinodales Calahorra 1602_page_4.png']
image_path8 = ['/content/image_folder8/Covarrubias - Tesoro lengua_page_7.png','/content/image_folder8/Covarrubias - Tesoro lengua_page_8.png','/content/image_folder8/Covarrubias - Tesoro lengua_page_9.png']
image_path9 = ['/content/image_folder9/Ezcaray - Vozes_page_8.png','/content/image_folder9/Ezcaray - Vozes_page_9.png','/content/image_folder9/Ezcaray - Vozes_page_10.png']
image_path10 = ['/content/image_folder10/Guardiola - Tratado nobleza_page_12.png','/content/image_folder10/Guardiola - Tratado nobleza_page_13.png','/content/image_folder10/Guardiola - Tratado nobleza_page_14.png']
image_path11 = ['/content/image_folder11/Mendo - Principe perfecto_page_14.png','/content/image_folder11/Mendo - Principe perfecto_page_15.png','/content/image_folder11/Mendo - Principe perfecto_page_16.png','/content/image_folder11/Mendo - Principe perfecto_page_17.png','/content/image_folder11/Mendo - Principe perfecto_page_18.png']
image_path12 = ['/content/image_folder12/Milán - El cortesano_page_6.png','/content/image_folder12/Milán - El cortesano_page_7.png']
image_path13 = ['/content/image_folder13/Paredes - Reglas generales_page_18.png','/content/image_folder13/Paredes - Reglas generales_page_19.png']
image_path14 = ['/content/image_folder14/R.12175.3 - Zamora_page_5.png','/content/image_folder14/R.12175.3 - Zamora_page_6.png']
image_path15 = ['/content/image_folder15/Recopilacion leyes 1640_page_11.png','/content/image_folder15/Recopilacion leyes 1640_page_12.png','/content/image_folder15/Recopilacion leyes 1640_page_13.png']


image_path.extend(image_path3)
image_path.extend(image_path4)
image_path.extend(image_path5)
image_path.extend(image_path6)
image_path.extend(image_path7)
image_path.extend(image_path8)
image_path.extend(image_path9)
image_path.extend(image_path10)
image_path.extend(image_path11)
image_path.extend(image_path12)
image_path.extend(image_path13)
image_path.extend(image_path14)
image_path.extend(image_path15)
print(image_path)

In [ ]:
import docx

# Step2
transcript = {}
def read_transcriptions(image_paths, file_path):
    # Load the DOCX file
    doc = docx.Document(file_path)

    # Initialize the dictionary to store text from each page

    # Extract text from each paragraph and assign it to the corresponding image path in the dictionary
    text = ''
    image_index = -1
    num_images = len(image_paths)

    for para in doc.paragraphs:
        stripped_text = para.text.strip()
        if "PDF" in stripped_text or "END OF EXTRACT" in stripped_text:
            if image_index >= 0:
                transcript[image_paths[image_index]] = text.strip()
            image_index += 1
            text = ''
        elif stripped_text != '':
            text += stripped_text + "\n"

    # Add the last page text if exists
    if text.strip() and image_index < num_images:
        transcript[image_paths[image_index]] = text.strip()

    return transcript


In [ ]:
transcription_path = "A/Padilla - 1 Nobleza virtuosa_Transcription (1).docx"
transcriptions = read_transcriptions(image_path1, transcription_path)  # Read transcriptions
transcription_path = "A/Padilla - 2 Noble perfecto_Transcription.docx"
transcriptions = read_transcriptions(image_path2, transcription_path)

# Dataset A
transcription_path = "A/Dataset A/PORCONES.82.4 – 1644 transcription.docx"
transcriptions = read_transcriptions(image_path3, transcription_path)
transcription_path = "A/Dataset A/PORCONES.954.2.45 transcription.docx"
transcriptions = read_transcriptions(image_path4, transcription_path)
transcription_path = "A/Dataset A/PORCONES.228.35 – 1636 transcription.docx"
transcriptions = read_transcriptions(image_path5, transcription_path)

# Dataset B

transcription_path = "A/Dataset B - different typeface/Buendia transcription.docx"
transcriptions = read_transcriptions(image_path6, transcription_path)
transcription_path = "A/Dataset B - different typeface/Constituciones sinodales transcription.docx"
transcriptions = read_transcriptions(image_path7, transcription_path)
transcription_path = "A/Dataset B - different typeface/Covarrubias transcription.docx"
transcriptions = read_transcriptions(image_path8, transcription_path)
transcription_path = "A/Dataset B - different typeface/Ezcaray transcription.docx"
transcriptions = read_transcriptions(image_path9, transcription_path)
transcription_path = "A/Dataset B - different typeface/Guardiola transcription.docx"
transcriptions = read_transcriptions(image_path10, transcription_path)
transcription_path = "A/Dataset B - different typeface/Mendo transcription.docx"
transcriptions = read_transcriptions(image_path11, transcription_path)
transcription_path = "A/Dataset B - different typeface/Milán transcription.docx"
transcriptions = read_transcriptions(image_path12, transcription_path)
transcription_path = "A/Dataset B - different typeface/Paredes transcription.docx"
transcriptions = read_transcriptions(image_path13, transcription_path)
transcription_path = "A/Dataset B - different typeface/R.12175.3 transcription.docx"
transcriptions = read_transcriptions(image_path14, transcription_path)
transcription_path = "A/Dataset B - different typeface/Recopilacion transcription.docx"
transcriptions = read_transcriptions(image_path15, transcription_path)


print(transcript.keys())

In [ ]:
# Without augmentations

import os
import re
import numpy as np
from skimage.io import imread
from skimage.color import rgb2gray

def get_segmented_image_paths(input_path, segmented_image_dir):
    # Get the base name from the input path
    base_name = os.path.basename(input_path)
    # print(base_name)
    # List all files in the directory
    files = os.listdir(segmented_image_dir)

    # Filter the files to get only those that start with the base name and end with "_line_*.png"
    segmented_files = [os.path.join(segmented_image_dir, f) for f in files if f.startswith(base_name) and "_line_" in f and f.endswith(".png")]

    return segmented_files

def extract_line_number(file_path):
    # Extract the line number(s) from the file name using regex
    match = re.search(r'_line_(\d+(_\d+)?).png$', file_path)
    if match:
        return list(map(int, match.group(1).split('_')))
    return [-1]

def sort_segmented_image_paths(image_paths):
    # Sort the list of paths based on the extracted line number
    sorted_paths = sorted(image_paths, key=extract_line_number)
    return sorted_paths

def align_segments_with_transcriptions(image_path, line_segmentation_path, transcription):
    segmented_image_paths = get_segmented_image_paths(image_path, line_segmentation_path)
    segmented_image_paths = sort_segmented_image_paths(segmented_image_paths)
    line_text = transcription.split('\n')

    count = 0
    data = []

    for path in segmented_image_paths:
        transcript = ""

        line_img = imread(path)[:,:,:3]

        if line_img.ndim > 2:
            line_img = rgb2gray(line_img)
        try:
            transcript = line_text[count]
            count += 1
            data.append((line_img, transcript))
            # print(count)
        except Exception as e:
            print(f"Error processing {image_path} - {path}: {e}")
            print(len(segmented_image_paths),len(line_text))
            print('\n')
            continue
    return data

# segmented_image_folder = "/content/drive/MyDrive/A/Line_Segmented_images"
segmented_image_folder = "A/New_Line_Segmented_images"
aligned_data = {}


for img_path in image_path:
    if img_path == '/content/image_folder8/Covarrubias - Tesoro lengua_page_8.png' or img_path == '/content/image_folder8/Covarrubias - Tesoro lengua_page_9.png':
        continue
    aligned_data[img_path] = align_segments_with_transcriptions(img_path, segmented_image_folder, transcript[img_path])


In [ ]:
total_line_segments = sum(len(data) for data in aligned_data.values())
print(f"Total line segment images: {total_line_segments}")

# Sample Training

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Trainer, TrainingArguments, EarlyStoppingCallback
from PIL import Image
import numpy as np
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
import torch.nn.functional as F
from evaluate import load
import albumentations as A
from transformers import get_cosine_schedule_with_warmup
import os

# Enable mixed precision training
torch.backends.cudnn.benchmark = True
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


# Load metrics
cer_metric = load("cer")
wer_metric = load("wer")

model_path = "qantev/trocr-large-spanish"
processor_path = "qantev/trocr-large-spanish"

processor = TrOCRProcessor.from_pretrained(processor_path, do_rescale=False,use_fast=True)
model = VisionEncoderDecoderModel.from_pretrained(model_path, use_safetensors=True)
# cache_dir="./cache"
# device_map="auto"

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = logits.argmax(-1)
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = []
    for label in labels:
        label_filtered = [token for token in label if token != -100]
        decoded_label = processor.tokenizer.decode(label_filtered, skip_special_tokens=True)
        decoded_labels.append(decoded_label)
    cer_score = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    wer_score = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"cer": cer_score, "wer": wer_score}

class LineDataset(Dataset):
    def __init__(self, processor, model, line_images, texts, target_size=(384, 96), max_length=512, apply_augmentation=True):
        self.line_images = line_images
        self.texts = texts
        self.processor = processor
        self.processor.image_processor.max_length = max_length
        self.processor.tokenizer.model_max_length = max_length
        self.model = model
        self.model.config.max_length = max_length
        self.target_size = target_size
        self.max_length = max_length
        self.apply_augmentation = apply_augmentation

        if apply_augmentation:
            self.transform = A.Compose([
                A.OneOf([
                    A.Rotate(limit=2, p=1.0),
                    A.ElasticTransform(alpha=0.3, sigma=50.0, alpha_affine=0.3, p=1.0),
                    A.OpticalDistortion(distort_limit=0.03, shift_limit=0.03, p=1.0),
                    A.CLAHE(clip_limit=2, tile_grid_size=(4, 4), p=1.0),
                    A.Affine(scale=(0.95, 1.05), translate_percent=(0.02, 0.02), shear=(-2, 2), p=1.0),
                    A.Perspective(scale=(0.01, 0.03), p=1.0),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                    A.GridDistortion(num_steps=3, distort_limit=0.02, p=1.0),
                    A.MedianBlur(blur_limit=3, p=1.0),
                ], p=0.7),
            ])
        else:
            self.transform = A.Compose([])

    def __len__(self):
        return len(self.line_images)

    def __getitem__(self, idx):
        image = self.line_images[idx]
        text = self.texts[idx]

        if isinstance(image, Image.Image):
            image = np.array(image)

        if image.ndim == 2:
            image = np.expand_dims(image, axis=-1)
            image = np.repeat(image, 3, axis=-1)

        image = (image * 255).astype(np.uint8)

        if self.apply_augmentation and self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        image = Image.fromarray(image)
        image = image.resize(self.target_size, Image.LANCZOS)
        image = np.array(image) / 255.0
        image = np.transpose(image, (2, 0, 1))

        encoding = self.processor(images=image, text=text, return_tensors="pt")
        encoding['labels'] = encoding['labels'][:, :self.max_length]
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        return encoding

def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = pad_sequence([item['labels'] for item in batch], batch_first=True, padding_value=-100)
    return {'pixel_values': pixel_values, 'labels': labels}

class SLiCTrainer(Trainer):
    def __init__(self, *args, label_smoothing=0.1, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_smoothing = label_smoothing
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):  # Added num_items_in_batch parameter
        labels = inputs.get("labels")
        pixel_values = inputs.get("pixel_values")
        lambda_reg = 0.005

        outputs = model(pixel_values=pixel_values, labels=labels)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        logits = logits.float()

        vocab_size = logits.size(-1)
        labels = torch.clamp(labels, min=0, max=vocab_size - 1)

        similarity_scores = self.calculate_similarity(logits, labels)
        slic_loss = self.compute_calibration_loss(logits, labels, similarity_scores)
        kl_loss = self.compute_kl_divergence(logits, labels)
        focal_loss = self.compute_focal_loss(logits, labels)

        total_loss = slic_loss + lambda_reg * (kl_loss + focal_loss)
        # total_loss = slic_loss + lambda_reg * (kl_loss)

        # Clear memory
        del similarity_scores
        torch.cuda.empty_cache()

        return (total_loss, outputs) if return_outputs else total_loss

    def calculate_similarity(self, logits, labels):
        batch_size, seq_len, vocab_size = logits.size()
        device = logits.device

        token_embeddings = self.model.get_output_embeddings().weight
        logits_softmax = F.softmax(logits, dim=-1)

        ru_ee_first_col = torch.zeros(vocab_size, 1, device=device)
        ru_ee_first_col[0] = 1.0

        similarity_scores = torch.matmul(logits_softmax, token_embeddings)
        similarity_scores = torch.matmul(similarity_scores, token_embeddings.t())
        ru_contribution = torch.matmul(logits_softmax, ru_ee_first_col)

        return (similarity_scores + ru_contribution) / 2

    def compute_calibration_loss(self, logits, labels, similarity_scores):
        batch_size, seq_len, vocab_size = logits.size()
        device = logits.device

        log_probs = F.log_softmax(logits, dim=-1)
        pos_samples = labels.unsqueeze(-1)
        neg_samples = torch.randint(0, vocab_size, (batch_size, seq_len, 1), device=device)

        l_rank = self.compute_rank_loss(log_probs, pos_samples, neg_samples)
        l_margin = self.compute_margin_loss(log_probs, similarity_scores, pos_samples, neg_samples)

        return l_rank + l_margin

    def compute_rank_loss(self, log_probs, pos_samples, neg_samples):
        beta = 0.15  # Increased from 0.1 for stronger ranking signal
        return torch.max(torch.zeros_like(log_probs[:, :, 0]),
                        beta - log_probs.gather(-1, pos_samples).squeeze(-1) +
                        log_probs.gather(-1, neg_samples).squeeze(-1)).mean()

    def compute_margin_loss(self, log_probs, similarity_scores, pos_samples, neg_samples):
        beta = 0.15  # Increased from 0.1 for stronger margin enforcement
        return torch.max(torch.zeros_like(log_probs[:, :, 0]),
                        beta * (similarity_scores.gather(-1, pos_samples).squeeze(-1) -
                               similarity_scores.gather(-1, neg_samples).squeeze(-1)) -
                        log_probs.gather(-1, pos_samples).squeeze(-1) +
                        log_probs.gather(-1, neg_samples).squeeze(-1)).mean()

    def compute_kl_divergence(self, logits, labels):
        log_probs = F.log_softmax(logits, dim=-1)
        target_probs = F.one_hot(labels, num_classes=logits.size(-1)).float()
        return F.kl_div(log_probs, target_probs, reduction='batchmean')

    def compute_focal_loss(self, logits, labels):
        ce_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), reduction='none')
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

def train_transformer_with_slic(line_images, texts, target_size=(384, 96), batch_size=16, max_length=512, val_split=0.15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    processor = TrOCRProcessor.from_pretrained(processor_path, do_rescale=False,use_fast=True)
    model = VisionEncoderDecoderModel.from_pretrained(model_path,use_safetensors=True)

    # Set model configurations
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
    model.config.max_length = max_length
    model.config.early_stopping = True
    model.config.no_repeat_ngram_size = 3
    model.config.length_penalty = 2.0
    model.config.num_beams = 4

    dataset = LineDataset(processor, model, line_images, texts, target_size, max_length, apply_augmentation=True)

    # Increased validation split for better evaluation
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    model = model.to(device)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=50,  # Increased epochs
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        logging_dir="./logs",
        logging_steps=100,
        save_steps=100,
        save_total_limit=3,
        eval_strategy="steps",
        eval_steps=100,
        learning_rate=3e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="cer",
        greater_is_better=False,
        logging_first_step=True,
        gradient_accumulation_steps=32,
        fp16=True,
        warmup_ratio=0.1,
        report_to = "none",
        # dataloader_num_workers=4,
        # dataloader_pin_memory=True,
        lr_scheduler_type="cosine_with_restarts",
        warmup_steps=5000,
        max_grad_norm=0.5,
        remove_unused_columns=False,
        label_smoothing_factor=0.1,
        adafactor=False,  # Using AdamW instead
        group_by_length=False,  # Speeds up training
    )

    optimizer = AdamW(model.parameters(),
                     lr=training_args.learning_rate,
                     weight_decay=0.01,
                     betas=(0.9, 0.999),
                     eps=1e-8)

    num_training_steps = len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    num_warmup_steps = 5000

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )

    trainer = SLiCTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        optimizers=(optimizer, scheduler),
        compute_metrics=compute_metrics,
        label_smoothing=0.1,
        gamma=2.0,
    )

    # Add early stopping with increased patience
    trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

    # Train the model
    trainer.train()

    # Save the final model and processor
    save_dir = "finetuned_transformer_model_vv2"
    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)

    return trainer

In [ ]:
line_images = []
texts = []
for page_data in aligned_data.values():
    for line_img, text in page_data:
        line_images.append(line_img)
        texts.append(text)

# Initialize training with corrected parameters
trainer = train_transformer_with_slic(
    line_images=line_images,
    texts=texts,
    target_size=(256, 64),
    batch_size=2,
    max_length=128,
    val_split=0.1
)

# Save the trained model
!mkdir -p model/finetuned_transformer_model


# Print final metrics
final_metrics = trainer.evaluate()
print("\nFinal Evaluation Metrics:")
print(f"Character Error Rate (CER): {final_metrics['eval_cer']:.4f}")
print(f"Word Error Rate (WER): {final_metrics['eval_wer']:.4f}")


# Flash Attention

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Trainer, TrainingArguments, EarlyStoppingCallback
from PIL import Image
import numpy as np
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
import torch.nn.functional as F
from evaluate import load
import albumentations as A
from transformers import get_cosine_schedule_with_warmup
import os

# Flash Attention imports
try:
    from flash_attn import flash_attn_func
    from flash_attn.bert_padding import unpad_input, pad_input
    FLASH_ATTENTION_AVAILABLE = True
    print("✓ Flash Attention available")
except ImportError:
    FLASH_ATTENTION_AVAILABLE = False
    print("⚠ Flash Attention not available. Install with: pip install flash-attn --no-build-isolation")

# Enable mixed precision training and Flash Attention optimizations
torch.backends.cudnn.benchmark = True
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Flash Attention configuration
if FLASH_ATTENTION_AVAILABLE:
    # Enable Flash Attention in transformers
    os.environ['TRANSFORMERS_USE_FLASH_ATTENTION'] = '1'
    # Set attention implementation
    torch.backends.cuda.enable_flash_sdp(True)

# Load metrics
cer_metric = load("cer")
wer_metric = load("wer")

model_path = "qantev/trocr-large-spanish"
processor_path = "qantev/trocr-large-spanish"

def load_model_with_flash_attention():
    """Load model with Flash Attention configuration"""
    processor = TrOCRProcessor.from_pretrained(processor_path, do_rescale=False, use_fast=True)
    
    if FLASH_ATTENTION_AVAILABLE:
        # Load model with Flash Attention configuration
        model = VisionEncoderDecoderModel.from_pretrained(
            model_path, 
            use_safetensors=True,
            torch_dtype=torch.float16,  # half precision for Flash Attention
            attn_implementation="flash_attention_2"  # Enable Flash Attention_2
        )
        print("✓ Model loaded with Flash Attention 2")
    else:
        model = VisionEncoderDecoderModel.from_pretrained(model_path, use_safetensors=True)
        print("⚠ Model loaded without Flash Attention")
    
    return processor, model

processor, model = load_model_with_flash_attention()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = logits.argmax(-1)
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = []
    for label in labels:
        label_filtered = [token for token in label if token != -100]
        decoded_label = processor.tokenizer.decode(label_filtered, skip_special_tokens=True)
        decoded_labels.append(decoded_label)
    cer_score = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    wer_score = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"cer": cer_score, "wer": wer_score}

class LineDataset(Dataset):
    def __init__(self, processor, model, line_images, texts, target_size=(384, 96), max_length=512, apply_augmentation=True):
        self.line_images = line_images
        self.texts = texts
        self.processor = processor
        self.processor.image_processor.max_length = max_length
        self.processor.tokenizer.model_max_length = max_length
        self.model = model
        self.model.config.max_length = max_length
        self.target_size = target_size
        self.max_length = max_length
        self.apply_augmentation = apply_augmentation

        if apply_augmentation:
            self.transform = A.Compose([
                A.OneOf([
                    A.Rotate(limit=2, p=1.0),
                    A.ElasticTransform(alpha=0.3, sigma=50.0, p=1.0),
                    A.OpticalDistortion(distort_limit=0.03, p=1.0),
                    A.CLAHE(clip_limit=2, tile_grid_size=(4, 4), p=1.0),
                    A.Affine(scale=(0.95, 1.05), translate_percent=(0.02, 0.02), shear=(-2, 2), p=1.0),
                    A.Perspective(scale=(0.01, 0.03), p=1.0),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                    A.GridDistortion(num_steps=3, distort_limit=0.02, p=1.0),
                    A.MedianBlur(blur_limit=3, p=1.0),
                ], p=0.7),
            ])
        else:
            self.transform = A.Compose([])

    def __len__(self):
        return len(self.line_images)

    def __getitem__(self, idx):
        image = self.line_images[idx]
        text = self.texts[idx]

        if isinstance(image, Image.Image):
            image = np.array(image)

        if image.ndim == 2:
            image = np.expand_dims(image, axis=-1)
            image = np.repeat(image, 3, axis=-1)

        image = (image * 255).astype(np.uint8)

        if self.apply_augmentation and self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        image = Image.fromarray(image)
        image = image.resize(self.target_size, Image.LANCZOS)
        image = np.array(image) / 255.0
        image = np.transpose(image, (2, 0, 1))

        encoding = self.processor(images=image, text=text, return_tensors="pt")
        encoding['labels'] = encoding['labels'][:, :self.max_length]
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        return encoding

def collate_fn(batch):
    """Optimized collate function for Flash Attention"""
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = pad_sequence([item['labels'] for item in batch], batch_first=True, padding_value=-100)
    
    # Convert to half precision if Flash Attention is available
    if FLASH_ATTENTION_AVAILABLE:
        pixel_values = pixel_values.half()
    
    return {'pixel_values': pixel_values, 'labels': labels}

class FlashAttentionSLiCTrainer(Trainer):
    def __init__(self, *args, label_smoothing=0.1, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_smoothing = label_smoothing
        self.gamma = gamma
        self.use_flash_attention = FLASH_ATTENTION_AVAILABLE

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        pixel_values = inputs.get("pixel_values")
        lambda_reg = 0.1

        # Use autocast for mixed precision with Flash Attention
        with torch.amp.autocast('cuda', enabled=self.use_flash_attention):
            outputs = model(pixel_values=pixel_values, labels=labels)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            
            # Ensure logits are in the right dtype
            if self.use_flash_attention:
                logits = logits.float()  # Convert back to float32 for loss computation

        vocab_size = logits.size(-1)
        labels = torch.clamp(labels, min=0, max=vocab_size - 1)

        # Compute custom losses with memory optimization
        with torch.amp.autocast('cuda', enabled=False):  # Disable autocast for custom loss computation
            similarity_scores = self.calculate_similarity(logits, labels)
            print("similarity_scores",similarity_scores)
            
            slic_loss = self.compute_calibration_loss(logits, labels, similarity_scores)
            kl_loss = self.compute_kl_divergence(logits, labels)

            print("KL Loss:",kl_loss)
            focal_loss = self.compute_focal_loss(logits, labels)
            print("Focal Loss:",focal_loss)    
        total_loss = slic_loss + lambda_reg * (kl_loss + focal_loss)

        # Memory cleanup
        del similarity_scores
        if self.use_flash_attention:
            torch.cuda.empty_cache()

        return (total_loss, outputs) if return_outputs else total_loss

    def calculate_similarity(self, logits, labels):
        batch_size, seq_len, vocab_size = logits.size()
        device = logits.device

        # Simplified similarity calculation to avoid dimension mismatch
        with torch.no_grad():
            token_embeddings = self.model.get_output_embeddings().weight
            token_embeddings = token_embeddings.float() #convert to float32
            logits_softmax = F.softmax(logits, dim=-1)

            ru_ee_first_col = torch.zeros(vocab_size, 1, device=device, dtype=torch.float32)
            ru_ee_first_col[0] = 1.0

            similarity_scores = torch.matmul(logits_softmax, token_embeddings)
            similarity_scores = torch.matmul(similarity_scores, token_embeddings.t())
            ru_contribution = torch.matmul(logits_softmax, ru_ee_first_col)

        return (similarity_scores + ru_contribution) / 2

    def compute_calibration_loss(self, logits, labels, similarity_scores):
        batch_size, seq_len, vocab_size = logits.size()
        device = logits.device

        log_probs = F.log_softmax(logits, dim=-1)
        pos_samples = labels.unsqueeze(-1)
        neg_samples = torch.randint(0, vocab_size, (batch_size, seq_len, 1), device=device)

        l_rank = self.compute_rank_loss(log_probs, pos_samples, neg_samples)
        l_margin = self.compute_margin_loss(log_probs, similarity_scores, pos_samples, neg_samples)

        print("Rank Loss",l_rank)
        print("Margin Loss",l_margin)

        return l_rank + l_margin

    def compute_rank_loss(self, log_probs, pos_samples, neg_samples):
        beta = 0.15
        return torch.max(torch.zeros_like(log_probs[:, :, 0]),
                        beta - log_probs.gather(-1, pos_samples).squeeze(-1) +
                        log_probs.gather(-1, neg_samples).squeeze(-1)).mean()

    def compute_margin_loss(self, log_probs, similarity_scores, pos_samples, neg_samples):
        beta = 0.15
        return torch.max(torch.zeros_like(log_probs[:, :, 0]),
                        beta * (similarity_scores.gather(-1, pos_samples).squeeze(-1) -
                               similarity_scores.gather(-1, neg_samples).squeeze(-1)) -
                        log_probs.gather(-1, pos_samples).squeeze(-1) +
                        log_probs.gather(-1, neg_samples).squeeze(-1)).mean()

    def compute_kl_divergence(self, logits, labels):
        log_probs = F.log_softmax(logits, dim=-1)
        target_probs = F.one_hot(labels, num_classes=logits.size(-1)).float()
        return F.kl_div(log_probs, target_probs, reduction='batchmean')

    def compute_focal_loss(self, logits, labels):
        ce_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), reduction='none')
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

def train_transformer_with_flash_attention(line_images, texts, target_size=(384, 96), batch_size=16, max_length=512, val_split=0.15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    
    # Load model and processor with Flash Attention
    processor, model = load_model_with_flash_attention()

    # Set model configurations
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
    model.config.max_length = max_length
    model.config.early_stopping = True
    model.config.no_repeat_ngram_size = 3
    model.config.length_penalty = 2.0
    model.config.num_beams = 4

    # Enable Flash Attention in model config if available
    if FLASH_ATTENTION_AVAILABLE:
        if hasattr(model.config, 'use_flash_attention_2'):
            model.config.use_flash_attention_2 = True
        # Optimize for memory efficiency
        model.gradient_checkpointing_enable()

    dataset = LineDataset(processor, model, line_images, texts, target_size, max_length, apply_augmentation=True)

    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    model = model.to(device)
    
    # Adjust batch size based on Flash Attention availability
    effective_batch_size = batch_size
    if FLASH_ATTENTION_AVAILABLE:
        # Flash Attention allows larger batch sizes
        effective_batch_size = min(batch_size * 2, 32)
        print(f"Using Flash Attention - increased batch size to {effective_batch_size}")

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=50,
        per_device_train_batch_size=effective_batch_size,
        per_device_eval_batch_size=1,
        logging_dir="./logs",
        logging_steps=100,
        save_steps=100,
        save_total_limit=3,
        eval_strategy="steps",
        eval_steps=100,
        learning_rate=3e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="cer",
        greater_is_better=False,
        logging_first_step=True,
        gradient_accumulation_steps=16 if FLASH_ATTENTION_AVAILABLE else 32,  # Reduced with Flash Attention
        fp16=False,
        bf16=FLASH_ATTENTION_AVAILABLE,  # Use BF16 with Flash Attention if available
        warmup_ratio=0.1,
        report_to="none",
        lr_scheduler_type="cosine_with_restarts",
        warmup_steps=5000,
        max_grad_norm=0.5,
        remove_unused_columns=False,
        label_smoothing_factor=0.1,
        adafactor=False,
        group_by_length=False,
        eval_accumulation_steps=4,
        # Flash Attention specific optimizations
        dataloader_num_workers=4 if FLASH_ATTENTION_AVAILABLE else 2,
        dataloader_pin_memory=True,
        torch_compile=FLASH_ATTENTION_AVAILABLE,  # Enable compilation with Flash Attention
        skip_memory_metrics=True,
    )

    # Optimizer with Flash Attention optimizations
    optimizer_kwargs = {
        'lr': training_args.learning_rate,
        'weight_decay': 0.01,
        'betas': (0.9, 0.999),
        'eps': 1e-8
    }
    
    if FLASH_ATTENTION_AVAILABLE:
        # Use fused optimizer if available
        try:
            from apex.optimizers import FusedAdam
            optimizer = FusedAdam(model.parameters(), **optimizer_kwargs)
            print("✓ Using FusedAdam optimizer")
        except ImportError:
            optimizer = AdamW(model.parameters(), **optimizer_kwargs)
            print("⚠ FusedAdam not available, using AdamW")
    else:
        optimizer = AdamW(model.parameters(), **optimizer_kwargs)

    # Learning rate scheduler
    num_training_steps = len(train_dataset) * training_args.num_train_epochs // (
        training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
    )
    num_warmup_steps = 5000

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )

    # Initialize trainer
    trainer = FlashAttentionSLiCTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        optimizers=(optimizer, scheduler),
        compute_metrics=compute_metrics,
        label_smoothing=0.1,
        gamma=2.0,
    )

    # Add early stopping
    trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

    print(f"\n🚀 Starting training with Flash Attention: {'Enabled' if FLASH_ATTENTION_AVAILABLE else 'Disabled'}")
    print(f"📊 Dataset size: {len(dataset)} samples")
    print(f"🔄 Training samples: {train_size}, Validation samples: {val_size}")
    print(f"💾 Effective batch size: {effective_batch_size}")
    print(f"⚡ Mixed precision: {'BF16' if FLASH_ATTENTION_AVAILABLE else 'FP16'}")

    # Train the model
    trainer.train()

    # Save the final model and processor
    save_dir = "finetuned_transformer_model_flash_attention"
    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)

    print(f"✅ Model saved to {save_dir}")
    return trainer

In [ ]:
line_images = []
texts = []
for page_data in aligned_data.values():
    for line_img, text in page_data:
        line_images.append(line_img)
        texts.append(text)

trainer = train_transformer_with_flash_attention(
        line_images=line_images,
        texts=texts,
        target_size=(256, 64),
        batch_size=4 if FLASH_ATTENTION_AVAILABLE else 2,
        max_length=128,
        val_split=0.1
)
    
# Evaluate final metrics
final_metrics = trainer.evaluate()
print("\n📈 Final Evaluation Metrics:")
print(f"Character Error Rate (CER): {final_metrics['eval_cer']:.4f}")
print(f"Word Error Rate (WER): {final_metrics['eval_wer']:.4f}")
    
